In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv("HF_TOKEN")

In [2]:
import torch  
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "AntonV/mamba2-130m-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", token=token)
model.eval()

/mnt/d/Hu_Module/Master/Semester 4/Study Project/Linear attention state management/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 218/218 [00:00<00:00, 235.72it/s]


Mamba2ForCausalLM(
  (backbone): Mamba2Model(
    (embeddings): Embedding(50288, 768)
    (layers): ModuleList(
      (0-23): 24 x Mamba2Block(
        (norm): Mamba2RMSNorm()
        (mixer): Mamba2Mixer(
          (act): SiLUActivation()
          (conv1d): Conv1d(1792, 1792, kernel_size=(4,), stride=(1,), padding=(3,), groups=1792)
          (in_proj): Linear(in_features=768, out_features=3352, bias=False)
          (norm): MambaRMSNormGated()
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): Mamba2RMSNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50288, bias=False)
)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(torch.__version__)
print(torch.version.cuda)

Using device: cuda
2.11.0+cu126
12.6


In [4]:
input_ids = tokenizer("Hey how are you doing?", return_tensors="pt")["input_ids"].to(device)
model.to(device)
output = model(input_ids)
print(output.cache_params)

DynamicCache(layers=[LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer])


In [5]:
output

Mamba2CausalLMOutput(loss=None, logits=tensor([[[-109.4034, -123.6546, -100.5854,  ..., -123.8113, -123.6327,
          -123.6537],
         [  12.6487,   -4.3741,   17.2194,  ...,   -4.1594,   -4.4720,
            -4.0875],
         [  61.6835,   47.2499,   65.0402,  ...,   47.3465,   47.1545,
            47.2795],
         [-101.0707, -118.8034,  -95.2718,  ..., -118.9732, -118.9620,
          -118.7094],
         [-106.0794, -124.9325, -100.0779,  ..., -125.0399, -124.9703,
          -124.7930],
         [ -25.4473,  -47.7910,  -26.4842,  ...,  -47.9525,  -47.9480,
           -47.8289]]], device='cuda:0', grad_fn=<UnsafeViewBackward0>), cache_params=DynamicCache(layers=[LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAtte

In [6]:
cache = output.cache_params

# Inspect the first layer
layer = cache.layers[0]
print(type(layer))
print(dir(layer))

# Print all attributes with shapes
for attr in vars(layer):
    val = getattr(layer, attr)
    if hasattr(val, 'shape'):
        print(f"{attr}: {val.shape}")
    elif isinstance(val, list):
        print(f"{attr}: list of {len(val)}")
    else:
        print(f"{attr}: {type(val)} = {val}")

<class 'transformers.cache_utils.LinearAttentionLayer'>
['__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', 'conv_kernel_size', 'conv_states', 'crop', 'device', 'dtype', 'has_previous_state', 'is_compileable', 'is_conv_states_initialized', 'is_recurrent_states_initialized', 'lazy_initialization', 'max_batch_size', 'offload', 'prefetch', 'recurrent_states', 'reorder_cache', 'reset', 'update_conv_state', 'update_recurrent_state']
conv_states: torch.Size([1, 1792, 4])
recurrent_states: torch.Size([1, 24, 64, 128])
is_conv_states_initialized: <class 'bool'> = True
is_recurrent_states_initialized

In [7]:
from Test import autoencoder, data, evaluate, model_loader, plot, state_utils, utils

In [8]:
config = utils.read_config("configs/config1.yaml")
    
paths = config["paths"]
output_dir = paths["output_dir"]
text_history_dir = paths["text_history_dir"]+"/history.txt"
state_dir = paths["state_dir"]+"/state.pt"
plot_dir = paths["plot_dir"]

In [9]:
dataset = data.load_data(config["data"]["name"], 
                    split=config["data"]["split"])
print(f"Dataset loaded with {len(dataset)} samples.")

Dataset loaded with 2011 samples.


In [10]:
sessions = data.extract_sessions(dataset)
states = []
session = sessions[0]
snapshots = data.build_turn_snapshots(session)

experiment_1_benchmark_path = output_dir + "/experiment_1_benchmark.csv"

In [ ]:
model.to("cpu")
for snap in snapshots:
    role = snap["role"]
    input_text = snap["new_text"]
    turn_id = snap["turn_id"]

    inputs = tokenizer(input_text, return_tensors="pt")
    with torch.no_grad():
        output = model(**inputs, use_cache=True, labels=inputs["input_ids"]).to("cpu")
    
    if role == "assistant":
        loss = output.loss.item() if output.loss is not None else 0.0
        ppl = torch.exp(torch.tensor(loss)).item() if loss > 0 else 0.0
        print(f"Turn {turn_id} - Perplexity: {ppl:.4f} - Loss : {loss:.4f}")
    
    cache = output.cache_params